In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    shapiro,
    pearsonr,
    spearmanr,
    mannwhitneyu,
    ttest_ind,
    ttest_rel,
    wilcoxon,
    chi2_contingency
)
MAIN_FOLDER = "../"
BILATERAL_FOLDER = "3-bilateral-2/"

In [ ]:
# ============================================================
# 1. Load dataset
# ============================================================
FILE_NAME = "paired_list"

# Read the excel file
paired = pd.read_excel("../../"+FILE_NAME+".xlsx")

# Show the head
print(paired.tail())

In [ ]:
# ============================================================
# 2. Create output folders
# ============================================================

output_folder = MAIN_FOLDER+BILATERAL_FOLDER
figures_folder = os.path.join(output_folder, "figures")
tables_folder = os.path.join(output_folder, "tables")

os.makedirs(output_folder, exist_ok=True)
os.makedirs(figures_folder, exist_ok=True)
os.makedirs(tables_folder, exist_ok=True)

In [ ]:
# Pell-Gregory symmetry
paired["pell_gregory_symmetric"] = (
    paired["pell_gregory_left"] == paired["pell_gregory_right"]
)

# Pederson symmetry
paired["pederson_symmetric"] = (
    paired["pederson_left"] == paired["pederson_right"]
)

# Winter angulation symmetry
paired["winter_angulation_symmetric"] = (
    paired["winter_angulation_left"] == paired["winter_angulation_right"]
)

# Full bilateral symmetry
paired["fully_symmetric"] = (
    paired["pell_gregory_symmetric"]
    & paired["pederson_symmetric"]
    & paired["winter_angulation_symmetric"]
)

In [ ]:
total_bilateral_patients = len(paired)

symmetry_summary = pd.DataFrame({
    "Symmetry Type": [
        "Pell-Gregory\nsymmetry",
        "Pederson\nsymmetry",
        "Winter angulation\nsymmetry",
        "Full bilateral\nsymmetry"
    ],
    "Number of Symmetric Patients": [
        paired["pell_gregory_symmetric"].sum(),
        paired["pederson_symmetric"].sum(),
        paired["winter_angulation_symmetric"].sum(),
        paired["fully_symmetric"].sum()
    ]
})

symmetry_summary["Total Bilateral Patients"] = total_bilateral_patients

symmetry_summary["Percentage (%)"] = (
    symmetry_summary["Number of Symmetric Patients"]
    / symmetry_summary["Total Bilateral Patients"]
    * 100
).round(2)

symmetry_summary

In [ ]:

new_row = pd.DataFrame([{'Symmetry Type': 'Pederson\ndifficulty group', 
                         'Number of Symmetric Patients': 278,
                         'Total Bilateral Patients': 400,
                         'Percentage (%)': 69.5
                        }])
df_sy = pd.concat([symmetry_summary, new_row], ignore_index=True)
df_sy

In [ ]:
df_sorted = df_sy.sort_values(by='Percentage (%)', ascending=False)
df_sorted['kappa'] = [0.43, 0.42, 0.44, 0.32, None]

In [ ]:
df_sorted

In [ ]:
# Save paired patient-level symmetry table
paired.to_excel(tables_folder+"/bilateral_symmetry_patient_level.xlsx", index=False)

# Save summary table
symmetry_summary.to_excel(tables_folder+"/bilateral_symmetry_summary.xlsx", index=False)

In [ ]:

plt.figure(figsize=(10, 6))

colors = plt.cm.Set3(np.linspace(0, 1, len(symmetry_summary)))

bars = plt.bar(
    symmetry_summary["Symmetry Type"],
    symmetry_summary["Percentage (%)"],
    color=colors
)

plt.title("Percentage of Bilateral Symmetry")
plt.xlabel("Symmetry Type")
plt.ylabel("Percentage of Patients (%)")
plt.xticks(rotation=30, ha="right")

for i, value in enumerate(symmetry_summary["Percentage (%)"]):
    plt.text(i, value + 1, f"{value:.1f}%", ha="center")

plt.tight_layout()
plt.savefig(figures_folder+"/bilateral_symmetry_bar_chart.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------
# Global style settings for academic appearance
# ---------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
    "font.size": 15,
    "axes.titlesize": 14,
    "axes.labelsize": 15,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 15,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

# ---------------------------
# Figure
# ---------------------------
fig, ax = plt.subplots(figsize=(9, 6))

#colors = plt.cm.Set3(np.linspace(0, 1, len(counts_nonzero)))
colors = plt.cm.Blues(np.linspace(0.45, 0.85, len(df_sorted["Percentage (%)"])))

bars = ax.bar(
    df_sorted["Symmetry Type"],
    df_sorted["Percentage (%)"],
    color=colors,
    edgecolor="black",
    linewidth=0.8
)

#ax.set_title("Percentage of Bilateral Symmetry")
ax.set_xlabel("Symmetry Type", labelpad=10)
ax.set_ylabel("Percentage (%)", labelpad=10)
#plt.xticks(rotation=0, ha="center")

# ---------------------------
# Axis formatting
# ---------------------------
ax.tick_params(axis="x", rotation=0)
for tick in ax.get_xticklabels():
    tick.set_horizontalalignment("center")

# Remove top and right spines for a cleaner academic look
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Add light horizontal gridlines
ax.yaxis.grid(True, linestyle="--", linewidth=0.6, alpha=0.5)
ax.set_axisbelow(True)

# ---------------------------
# Add values on top of bars
# ---------------------------
y_max = df_sorted["Percentage (%)"].values.max()
ax.set_ylim(0, y_max * 1.15)

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + (y_max * 0.02),
        f"{height}%",
        ha="center",
        va="bottom",
        fontsize=15,
        fontweight="bold"
    )

# Add kappa values inside bars
for bar, kappa in zip(bars, df_sorted["kappa"]):
    if pd.notna(kappa):
        height = bar.get_height()

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height * 0.50,
            f"κ = {kappa:.2f}",
            ha="center",
            va="center",
            fontsize=15,
            color="white",
            fontweight="bold"
        )
# ---------------------------
# Layout and save
# ---------------------------
plt.tight_layout()

bar_path = os.path.join(figures_folder, "bilateral_symmetry_bar_chart.png")
plt.savefig(
    bar_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close()

print(f"Saved: {bar_path}")

In [ ]:
full_symmetry_counts = paired["fully_symmetric"].value_counts()

full_symmetry_counts.index = full_symmetry_counts.index.map({
    True: "Fully Symmetric",
    False: "Not Fully Symmetric"
})

def my_autopct(pct):
    return ("%1.1f%%" % pct) if pct > 1.0 else ""



In [ ]:
plt.figure(figsize=(8, 8))

colors = plt.cm.Pastel1(np.linspace(0, 1, len(full_symmetry_counts)))

wedges, texts, autotexts = plt.pie(
    full_symmetry_counts,
    autopct=my_autopct,
    startangle=90,
    pctdistance=0.85,
    colors=colors,
    wedgeprops=dict(width=0.5, edgecolor="w")
)

plt.title("Percentage of Fully Bilateral Symmetric Patients")

plt.legend(
    wedges,
    full_symmetry_counts.index.astype(str),
    title="Symmetry Status",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1)
)

plt.tight_layout()
plt.savefig(MAIN_FOLDER+"bilateral/fully_bilateral_symmetric_donut_chart.png", dpi=300, bbox_inches="tight")
plt.show()